# 8.2 Cube 性能（上）：K 切分原理

Cube 只干一件事：矩阵乘 `C[M,N] = A[M,K] × B[K,N]`。FA kernel 里的两个 mm 的 K 轴长度天差地别，**K 切分是 Cube 侧最核心的性能变量**。

## 1. 三条轴，三种角色

对 `C[M,N] = A[M,K] × B[K,N]`：

| 轴 | 角色 | 切它的后果 |
|--|--|--|
| M（A 的行） | 输出行 | 每块独立出结果，无累加，L0C 按块分占 |
| N（B 的列） | 输出列 | 同 M，独立出结果 |
| **K** | **累加轴** | **切了不能独立出结果，必须做部分和** |

M/N 切块只是「分而治之」；**K 切块是「分期付款」**——每一段算出一个部分和，最后要把所有段的部分和加起来才是真结果。

## 2. FA 的两个 mm，K 轴处境完全不同

```
mm1: S[M, S2] = Q[M, D] × K^T[D, S2]     # K 轴 = D（head 维，典型 128）
mm2: O[M, D]  = P[M, S2] × V[S2, D]      # K 轴 = S2（KV 序列长）
```

- **mm1**：K = D = 128（fp16 下 128×16 分形 ×8），通常**一次全载**，不需要切；
- **mm2**：K = S2，训练可到 8K+，推理 KV Cache 累积可上万，**必须切**。

这正是 FA kernel 的天然结构：外层循环每推进一个 S2 基本块，就是 mm2 沿 K 轴再「付款」一段，同时触发一次 softmax update。

## 3. L0 容量：切的根本原因

Cube 的数据必须先进 L0（L0A 放左矩阵、L0B 放右矩阵、L0C 放输出），容量都是 KB 级：

```
L0A: mBlocks × singleCoreM × singleCoreK × dtypeSize ≤ L0A_SIZE
L0B: nBlocks × singleCoreK × singleCoreN × dtypeSize ≤ L0B_SIZE
L0C: mBlocks × nBlocks × singleCoreM × singleCoreN × dtypeSize ≤ L0C_SIZE
```

注意一个关键不对称：**K 切小只能省 L0A/L0B，省不了 L0C**——输出 `M×N` 的大小与 K 无关。所以当输出块已经把 L0C 占满时，K 切得再小也没有额外收益。

## 4. K 切分对带宽的影响

K 切分带来三笔相互冲突的账：

| 影响 | 方向 | 原因 |
|--|--|--|
| MTE1 整块率 | **变差** | K 块变多，非 16 对齐的尾块概率上升，小搬运多 |
| A/B 复用率 | **变差** | 一段 A 装入 L0 后，被 M/N 方向分形复用的窗口变短 |
| 部分和开销 | **新增** | 每段输出 L0C 都要 fixup（部分和回读/累加） |

**总量不变、效率变差**——K 切分省的是容量，花的是带宽效率。这就是「能不切就不切，必须切就切得尽量大」的定量依据（8.3 展开）。

## 5. 图示

![K 切分与部分和](images/k_split.svg)

## 动手实验：K 切分对搬运效率的影响

下面用 numpy 模拟：固定 `M=128, N=128, K=4096`（fp16，分形 16），从「K 全载」到「K 切成 16」逐档观察 MTE1 搬运笔数、整块率和复用率的变化。


In [ ]:
# 动手：K 切分对搬运效率的影响（纯建模模拟，不依赖昇腾环境）
import numpy as np

M, N, K = 128, 128, 4096      # mm2 的典型形态：M=S1块, N=D, K=S2
dtype_size = 2                # fp16
FRACTAL = 16                  # 昇腾矩阵分形基元 16x16
L0A_SIZE = 64 * 1024          # L0A 容量（示意值）
L0B_SIZE = 64 * 1024

print(f"{'kBase':>6} {'L0A占用':>9} {'能否全载':>8} {'搬运笔数':>8} {'尾块浪费':>9} {'A复用率':>8}")
for k_base in [4096, 2048, 1024, 512, 256, 128, 64, 32, 16]:
    # L0A 占用：M×kBase；L0B 占用：kBase×N
    l0a = M * k_base * dtype_size
    l0b = k_base * N * dtype_size
    fit = (l0a <= L0A_SIZE) and (l0b <= L0B_SIZE)

    # 搬运笔数：沿 K 切成 K/k_base 段，每段 A、B 各搬运一次
    segs = K // k_base if K % k_base == 0 else K // k_base + 1
    moves = 2 * segs

    # 尾块浪费：每段里 k_base 对齐到 16 后的填充开销（这里 k_base 全取 16 的倍数）
    k_align = ((k_base + FRACTAL - 1) // FRACTAL) * FRACTAL
    waste = (k_align * segs - K) / K

    # A 复用率：一段 A 装入 L0 后，能被多少个 N 方向分形使用
    # 全载时 N 方向有 N/16 个分形都复用这同一段 A
    n_fractals = N // FRACTAL
    reuse = n_fractals if k_base == K else max(1, n_fractals)

    print(f"{k_base:>6} {l0a:>8}B {str(fit):>8} {moves:>8} {waste:>8.1%} {reuse:>8}")


### 观察结论

1. **K 全载（kBase=4096）**：搬运笔数最少、复用率最高——只要容量装得下，全载永远最优；
2. **K 切小**：搬运笔数线性上涨、每段装载的 A 被 N 方向分形复用的窗口被稀释；
3. **L0A/L0B 占用与 kBase 成正比**：容量装不下时才被迫切小。

这也解释了 mm1（K=D=128）为什么几乎不讨论 K 切分：128 一段在 L0 里毫无压力。

## 6. 部分和：K 切分新增的第三笔账

K 不切时，L0C 里一次就能得到完整结果；K 切成 n 段后：

```
第 1 段: L0C = A1×B1
第 2 段: L0C += A2×B2    ← 需要先读旧值再累加（fixup）
...
第 n 段: L0C += An×Bn
```

每一段多出来的「读旧值 + 加法 + 写回」就是 **fixup 开销**，段数越多 fixup 越多。部分产品形态支持 L0C 原地累加（`enAcc` 类开关），可以省掉回读，但段数本身的调度开销仍然在。

## 本节要点

- K 是**累加轴**：切 K 必然引入部分和；切 M/N 不会；
- FA 的 mm2（K=S2）是 K 切分的主战场，mm1（K=D）通常全载；
- K 切小省 L0A/L0B 容量，但花掉整块率、复用率和 fixup——**切分是容量和带宽的交换**；
- L0C 占用与 K 无关，输出块占满 L0C 后继续切 K 没有收益。

## 小测验

1. 为什么说「切 M/N 是分而治之，切 K 是分期付款」？
2. mm2 的 K 轴是哪一个语义维度？为什么它决定了 K 切分在 FA 里不可避免？
3. K 切小之后，L0C 的占用会下降吗？为什么？
4. 列出 K 切分带来的三笔「变差/新增」的开销。

>>> 参考答案见 [answer/08.02_answer.txt](answer/08.02_answer.txt)
